# Week_2_a
This notebook involves understanding how pre-trained tranformer models are loaded, tokenized, and used for inference.

The first experimentation, as below, fulfils this in a Python Notebook, and will form the basis of further experimentation for hands-on implementation, testing and debugging, and weekly review.

In [1]:
pip install torch transformers

In [4]:
from transformers import pipeline

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    "sentiment-analysis",
    model=model_name
)

text = "I am not sure about learning transformers in terms of career development."

result = classifier(text)

print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9994475245475769}]


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "I really enjoyed learning how transformer inference works."

tokens = tokenizer.tokenize(text)

print(tokens)

['i', 'really', 'enjoyed', 'learning', 'how', 'transform', '##er', 'inference', 'works', '.']


In [11]:
encoded = tokenizer(
    text,
    return_tensors="pt"
)

print(encoded)

{'input_ids': tensor([[  101,  1045,  2428,  5632,  4083,  2129, 10938,  2121, 28937,  2573,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [12]:
print("Tokens:")
print(tokens)

print("\nInput IDs:")
print(encoded["input_ids"])

print("\nAttention mask:")
print(encoded["attention_mask"])

Tokens:
['i', 'really', 'enjoyed', 'learning', 'how', 'transform', '##er', 'inference', 'works', '.']

Input IDs:
tensor([[  101,  1045,  2428,  5632,  4083,  2129, 10938,  2121, 28937,  2573,
          1012,   102]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [13]:
ids = encoded["input_ids"][0]

print(
    tokenizer.convert_ids_to_tokens(ids)
)

['[CLS]', 'i', 'really', 'enjoyed', 'learning', 'how', 'transform', '##er', 'inference', 'works', '.', '[SEP]']


In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name
)

print(model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [15]:
print(model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



In [16]:
print(model.config.id2label)

{0: 'NEGATIVE', 1: 'POSITIVE'}


In [17]:
import torch

with torch.no_grad():
    outputs = model(**encoded)

print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[-4.0110,  4.2672]]), hidden_states=None, attentions=None)


In [18]:
logits = outputs.logits

print(logits)
print(logits.shape)

tensor([[-4.0110,  4.2672]])
torch.Size([1, 2])


In [19]:
probabilities = torch.softmax(logits, dim=-1)

print(probabilities)

tensor([[2.5393e-04, 9.9975e-01]])


In [20]:
predicted_class = probabilities.argmax(dim=-1).item()

label = model.config.id2label[predicted_class]

print("Prediction:", label)
print("Probability:", probabilities[0][predicted_class].item())

Prediction: POSITIVE
Probability: 0.9997460246086121


In [21]:
pipeline_result = classifier(text)

print("Pipeline:")
print(pipeline_result)

print("\nManual:")
print({
    "label": label,
    "score": probabilities[0][predicted_class].item()
})

Pipeline:
[{'label': 'POSITIVE', 'score': 0.9997460246086121}]

Manual:
{'label': 'POSITIVE', 'score': 0.9997460246086121}


In [23]:
examples = [
    "This course is fantastic.",
    "I hated every minute of this movie.",
    "The product arrived yesterday.",
    "The software is incredibly frustrating to use.",
    "The new implementation works much better.",
    "The product arrived yesterday."
]

for text in examples:
    print(text)
    print(classifier(text))
    print()

This course is fantastic.
[{'label': 'POSITIVE', 'score': 0.999882698059082}]

I hated every minute of this movie.
[{'label': 'NEGATIVE', 'score': 0.9996154308319092}]

The product arrived yesterday.
[{'label': 'POSITIVE', 'score': 0.9814155101776123}]

The software is incredibly frustrating to use.
[{'label': 'NEGATIVE', 'score': 0.9995191097259521}]

The new implementation works much better.
[{'label': 'POSITIVE', 'score': 0.9710241556167603}]

The product arrived yesterday.
[{'label': 'POSITIVE', 'score': 0.9814155101776123}]

